In [0]:
%run "./librerias"

In [0]:
%run "./configuration"

In [0]:
def add_ingestion_date (input_df):
    output_df = input_df.withColumn("ingestion_date", current_timestamp())
    return output_df

#####add_ingestion_date (input_df)

In [0]:
def add_env (input_df):
    output_df = input_df.withColumn("enviroment", lit("produccion"))
    return output_df

#####add_env (input_df)

In [0]:
def add_file_date (input_df):
    output_df = input_df.withColumn("file_date", lit(v_file_date))
    return output_df

#####add_file_date (input_df)

In [0]:
def delete_table (input_table):
    if spark.catalog.tableExists(input_table):
        spark.sql(f"delete from {input_table} where file_date = '{file_date}'")
        return f"Table {input_table} existe, data '{file_date}' borrada"
    else :
        return f"Table {input_table} no existe"
 


In [0]:
def overwrite_partition (dbname, table_name, column_partition, file_date):
    if spark.catalog.tableExists(f"{dbname}.{table_name}"):
        spark.sql(f"delete from {dbname}.{table_name} where {column_partition} = '{file_date}'")
        return f"Table {dbname}.{table_name} existe, se borraron registros de la partition {column_partition} = {file_date}"
    else :
        return f"Table {dbname}.{table_name} no existe"

In [0]:
def merge_delta_lake (dbname, table_name, df_imput, merge_condition, column_partition):
    if spark.catalog.tableExists(f"{dbname}.{table_name}"):
        deltaTable = DeltaTable.forName(spark, f"{dbname}.{table_name}")
        deltaTable.alias("target")\
                  .merge(
                        df_imput.alias("source"),
                        f"{merge_condition} and target.{column_partition} = source.{column_partition}" 
                      )\
                  .whenMatchedUpdateAll()\
                  .whenNotMatchedInsertAll()\
                  .execute()
        return f"Table {dbname}.{table_name}, se actualizaron registros de la partition {column_partition} = {v_file_date}"
    else:
        df_imput.write.mode("overwrite").partitionBy(column_partition).format("delta").saveAsTable(f"{dbname}.{table_name}")
        return f"Table {dbname}.{table_name}, se insertaron registros de la partition {column_partition} = {v_file_date}"